# Colab GPU Trong IDE (Git Sync): Train LoRA Minh Khang + Giọng Biểu Cảm + Video Hành Động

1. **Cell 1:** Pull code & 12 ảnh mới nhất từ GitHub (`nguyenkhanhan123/Test`) + kiểm tra GPU.
2. **Cell 2:** Train SDXL LoRA (`pytorch_lora_weights.safetensors`) — **tự bỏ qua nếu đã có file LoRA**.
3. **Cell 2B:** Sao lưu LoRA sang Google Drive (nếu đã mount) + kéo bản vá Git mới.
4. **Cell 3:** Dùng LoRA vừa train sinh Keyframe hành động mới cho `shot_103`, `shot_104`.
5. **Cell 4:** Audio-First — tạo giọng biểu cảm & khóa thời lượng từng shot.
6. **Cell 5:** Render cảnh hành động AI (`Wan 2.2` trên L4, hoặc `SVD-XT fp16` gọn nhẹ trên T4).
7. **Cell 6:** Ghép video hoàn chỉnh + **mở giao diện Gradio (`share=True`) để xem và tải video + LoRA về máy**.
8. **Cell 7:** Bấm chạy ô này khi bạn đã xem & tải xong để tắt GPU tiết kiệm CU.

In [ ]:
# CELL 1: ĐỒNG BỘ GIT + KIỂM TRA GPU + CÀI THƯ VIỆN CỐT LÕI
import os
REPO_URL = 'https://github.com/nguyenkhanhan123/Test.git'
REPO_DIR = '/content/Test'
PROJECT_DIR = '/content/Test/ai_film_pipeline'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin main && git reset --hard origin/main

%cd {PROJECT_DIR}
!nvidia-smi
!pip uninstall -y -q torchao
!pip install -q --upgrade edge-tts pysubs2 pydantic imageio-ffmpeg mutagen gradio peft bitsandbytes accelerate transformers sentencepiece git+https://github.com/huggingface/diffusers.git
!python prepare_user_assets.py --trigger nv_minhkhang

In [ ]:
# CELL 2: TRAIN CHARACTER LORA (CHỈ TRAIN 1 LẦN — TỰ BỎ QUA NẾU ĐÃ CÓ FILE LORA)
%cd /content/Test/ai_film_pipeline
import os, shutil

local_lora = 'loras/nv_minhkhang_lora_v1/pytorch_lora_weights.safetensors'
drive_lora = '/content/drive/MyDrive/ai_film_loras/nv_minhkhang_lora_v1.safetensors'
FORCE_RETRAIN = False  # Đổi thành True nếu bạn thay bộ ảnh nhân vật mới

if not os.path.exists(local_lora) and os.path.exists(drive_lora):
    os.makedirs(os.path.dirname(local_lora), exist_ok=True)
    shutil.copy2(drive_lora, local_lora)
    print('[LoRA] Đã khôi phục LoRA từ Google Drive:', local_lora)

if os.path.exists(local_lora) and not FORCE_RETRAIN:
    print(f'[LoRA] Đã có sẵn {local_lora} ({os.path.getsize(local_lora)/1e6:.2f} MB) -> Bỏ qua bước train!')
else:
    !python cloud_scripts/train_sdxl_lora_peft.py \
      --dataset user_inputs/photos \
      --output loras/nv_minhkhang_lora_v1 \
      --steps 360 \
      --resolution 768

In [ ]:
# CELL 2B: SAO LƯU LORA SANG DRIVE (NẾU ĐÃ MOUNT) + KÉO CODE MỚI NHẤT
%cd /content/Test/ai_film_pipeline
import os, shutil

local_lora = '/content/Test/ai_film_pipeline/loras/nv_minhkhang_lora_v1/pytorch_lora_weights.safetensors'
if os.path.exists(local_lora) and os.path.exists('/content/drive/MyDrive'):
    os.makedirs('/content/drive/MyDrive/ai_film_loras', exist_ok=True)
    dst = '/content/drive/MyDrive/ai_film_loras/nv_minhkhang_lora_v1.safetensors'
    shutil.copy2(local_lora, dst)
    print(f'[BACKUP OK] Đã lưu LoRA sang Drive: {dst} ({os.path.getsize(dst)/1e6:.2f} MB)')
elif os.path.exists(local_lora):
    print(f'[LOCAL LORA OK] {local_lora} ({os.path.getsize(local_lora)/1e6:.2f} MB) — Cell 6 sẽ mở link Gradio cho bạn tải thẳng về máy!')

!git -C /content/Test fetch origin main && git -C /content/Test reset --hard origin/main
!pip uninstall -y -q torchao

In [ ]:
# CELL 3: DÙNG LORA VỪA TRAIN SINH KEYFRAME HÀNH ĐỘNG MỚI CHO SHOT_103 & SHOT_104
%cd /content/Test/ai_film_pipeline
!pip uninstall -y -q torchao
!python cloud_scripts/step4_gen_keyframes_lora.py

In [ ]:
# CELL 4: AUDIO-FIRST — TẠO GIỌNG BIỂU CẢM & KHÓA THỜI LƯỢNG TỪNG SHOT
%cd /content/Test/ai_film_pipeline
from pathlib import Path
voices_ready = list(Path('user_inputs/voices').glob('*.wav'))
if voices_ready:
    !pip install -q git+https://github.com/danhtran2mind/F5-TTS.git whisperx
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='danhtran2mind/Vi-F5-TTS', local_dir='/content/Vi-F5-TTS')
    from cloud_scripts.step3_cloud_vi_f5_tts import clone_episode_voices_vi_f5
    clone_episode_voices_vi_f5(Path('.'), Path('script_action_demo.json'), Path('/content/Vi-F5-TTS'))
else:
    print('Dùng Edge TTS biểu cảm đa sắc thái (thì thầm / căng thẳng / hô chiến đấu / lắng dịu)...')
    !python step_action_demo.py --tts-only

In [ ]:
# CELL 5: RENDER VIDEO HÀNH ĐỘNG AI (WAN 2.2 TRÊN L4 HOẶC SVD-XT FP16 TRÊN T4)
%cd /content/Test/ai_film_pipeline
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python cloud_scripts/step5_batch_wan22_diffusers.py --script script_action_demo.json --steps 24

In [ ]:
# CELL 6: GHÉP VIDEO HOÀN CHỈNH + MỞ LINK GRADIO XEM & TẢI VỀ MÁY
%cd /content/Test/ai_film_pipeline
import os, shutil, subprocess
subprocess.run(['python', 'step_action_demo.py', '--skip-tts'], check=True)

video_out = 'output/action_pilot_demo.mp4'
lora_out = 'loras/nv_minhkhang_lora_v1/pytorch_lora_weights.safetensors'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs('/content/drive/MyDrive/ai_film_loras', exist_ok=True)
    shutil.copy2(video_out, '/content/drive/MyDrive/ai_film_loras/action_pilot_demo.mp4')
    if os.path.exists(lora_out):
        shutil.copy2(lora_out, '/content/drive/MyDrive/ai_film_loras/nv_minhkhang_lora_v1.safetensors')
    print('[DRIVE SAVED] MyDrive/ai_film_loras/action_pilot_demo.mp4')

!pip install -q gradio
import gradio as gr
with gr.Blocks(title='AI Action Video Demo') as demo:
    gr.Markdown('## Video Hành Động Hoàn Chỉnh + Ảnh LoRA Đã Train')
    gr.Video(value=video_out, label='Video Hoàn Chỉnh (action_pilot_demo.mp4)')
    with gr.Row():
        if os.path.exists('assets/keyframes_lora/shot_103.png'):
            gr.Image(value='assets/keyframes_lora/shot_103.png', label='Keyframe LoRA shot_103')
        if os.path.exists('assets/keyframes_lora/shot_104.png'):
            gr.Image(value='assets/keyframes_lora/shot_104.png', label='Keyframe LoRA shot_104')
    dl_files = [p for p in [video_out, lora_out] if os.path.exists(p)]
    gr.File(value=dl_files, label='Bấm vào đây để tải Video (.mp4) & LoRA (.safetensors) về máy')
demo.launch(share=True)

In [ ]:
# CELL 7: TẮT GPU COLAB (CHỈ BẤM SAU KHI ĐÃ XEM & TẢI XONG VIDEO Ở CELL 6)
from google.colab import runtime
runtime.unassign()